In [1]:
# ============================================================
# FINAL SCALED_2025 — 2025 TARGETS + SEASONAL HYDRO ECONOMICS
# ============================================================

import pypsa
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn = root / "resources" / "networks" / "base_s_50_elec.nc"
out_fn  = root / "results" / "networks" / "base_s_50_elec_scaled_2025.nc"

n = pypsa.Network(base_fn)
print("Loaded base network:", base_fn.name)

# ------------------------------------------------------------
# COUNTRIES
# ------------------------------------------------------------
MAIN_COUNTRIES = ["DK", "NL", "BE", "DE", "FR", "NO", "PL", "SE", "PT"]

bus_country = n.buses.country

# ------------------------------------------------------------
# FIX 0 — NETCDF SAFETY (CRITICAL)
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# ------------------------------------------------------------
# FIX 1 — DISPATCH-ONLY (NO HUB, NO EXPANSION)
# ------------------------------------------------------------
for df_name in ["generators", "links", "lines", "storage_units"]:
    if hasattr(n, df_name):
        df = getattr(n, df_name)
        if "p_nom_extendable" in df.columns:
            df["p_nom_extendable"] = False
if hasattr(n, "lines") and "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

# ------------------------------------------------------------
# FIX 2 — HYDRO STORAGE PHYSICS (ENERGY CONSTRAINTS)
# ------------------------------------------------------------
hydro_carrier = n.storage_units.carrier.str.lower()

# Reservoirs: seasonal energy buffers
n.storage_units.loc[hydro_carrier.str.contains("reservoir"), "max_hours"] = 3000

# Pumped hydro: daily / weekly shifting
n.storage_units.loc[hydro_carrier.str.contains("phs|pumped"), "max_hours"] = 100

# No cyclic cheating
n.storage_units["cyclic_state_of_charge"] = False

# ------------------------------------------------------------
# FIX 3 — HYDRO INFLOWS VIA FLH (OPTION A — CORRECT)
# ------------------------------------------------------------
HYDRO_RES_FLH = {
    "NO": 5200,
    "SE": 4800,
    "FR": 3200,
    "PT": 3000,
    "PL": 2600,
    "DE": 2200,
    "BE": 1800,
    "NL": 1500,
    "DK": 1500,
}

res_mask = hydro_carrier.str.contains("reservoir")

for su, row in n.storage_units.loc[res_mask].iterrows():
    country = bus_country.loc[row.bus]
    if country not in HYDRO_RES_FLH:
        continue

    flh = HYDRO_RES_FLH[country]
    target_energy = row.p_nom * flh

    inflow = n.storage_units_t.inflow[su]
    current_energy = inflow.sum()

    if current_energy > 0:
        n.storage_units_t.inflow[su] *= target_energy / current_energy

print("✓ Hydro inflows scaled to FLH targets")

# ------------------------------------------------------------
# FIX 4 — SEASONAL WATER VALUE (ECONOMIC CORE)
# ------------------------------------------------------------
# €/MWh opportunity cost of releasing water
WATER_VALUE = {
    "winter": 40.0,
    "spring": 20.0,
    "summer": 5.0,
    "autumn": 25.0,
}

def season(ts):
    m = ts.month
    if m in [12, 1, 2]:
        return "winter"
    if m in [3, 4, 5]:
        return "spring"
    if m in [6, 7, 8]:
        return "summer"
    return "autumn"

# Build time-dependent marginal cost for hydro
hydro_mask = n.storage_units.carrier.str.lower().str.contains("hydro")

mc = pd.DataFrame(
    0.0,
    index=n.snapshots,
    columns=n.storage_units.index,
)

for t in n.snapshots:
    s = season(pd.Timestamp(t))
    mc.loc[t, hydro_mask] = WATER_VALUE[s]

# Assign to network (PyPSA ≥ 0.31 compatible)
n.storage_units_t.marginal_cost = mc

print("✓ Seasonal hydro water value applied")

# ------------------------------------------------------------
# FIX 5 — LOAD SHEDDING (LAST RESORT)
# ------------------------------------------------------------
LS_COST = 10_000.0

if "load_shedding" not in n.carriers.index:
    n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

missing_ls = [b for b in n.buses.index if f"load_shed_{b}" not in n.generators.index]
for b in missing_ls:
    n.add(
        "Generator",
        name=f"load_shed_{b}",
        bus=b,
        carrier="load_shedding",
        p_nom=1e6,
        marginal_cost=LS_COST,
    )

print("✓ Load shedding generators ensured")

# ============================================================
# NETCDF SAFETY — CLEAN CARRIER METADATA (MANDATORY)
# ============================================================

# Ensure carriers.color is string-only
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# Ensure carriers.nice_name is string-only
if "nice_name" in n.carriers.columns:
    n.carriers["nice_name"] = n.carriers["nice_name"].astype(str)

# Replace common invalid placeholders
n.carriers["color"] = n.carriers["color"].replace(
    {"0": "#000000", "nan": "#000000", "None": "#000000"}
)

print("✓ Carrier metadata cleaned for NetCDF export")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn.parent.mkdir(parents=True, exist_ok=True)
n.export_to_netcdf(out_fn)

print("✓ FINAL scaled_2025 saved to:")
print(out_fn)


INFO:pypsa.io:Imported network base_s_50_elec.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded base network: base_s_50_elec.nc
✓ Hydro inflows scaled to FLH targets
✓ Seasonal hydro water value applied
✓ Load shedding generators ensured
✓ Carrier metadata cleaned for NetCDF export


INFO:pypsa.io:Exported network 'base_s_50_elec_scaled_2025.nc' contains: generators, links, loads, storage_units, lines, carriers, buses, stores


✓ FINAL scaled_2025 saved to:
C:\Users\franc\pypsa\pypsa-eur\results\networks\base_s_50_elec_scaled_2025.nc
